In [6]:
import dill
import os
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from xgboost import XGBClassifier

In [2]:
PROJ_PATH = Path(os.getcwd()).parent
DATA_PATH = PROJ_PATH / 'Data'

full_dataset = pd.read_csv(DATA_PATH / 'full_dataset.csv')
X_full = full_dataset.copy().iloc[:, :-1]
y_full = full_dataset.copy()['UPSET']

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=3
)

In [3]:
xgb_param_grid = {
    'max_depth': [1, 2, 3],
    'n_estimators': range(5, 16),
    'reg_lambda': [0.001, 0.01, 0.1, 0.2, 0.3, 0.4]
}
xgb_grid = GridSearchCV(
    estimator=XGBClassifier(learning_rate=0.2, colsample_bytree=0.7, random_state=123),
    param_grid=xgb_param_grid,
    cv=cv,
    scoring='accuracy',
    refit=True,
    n_jobs=1
)
xgb_grid.fit(X_full, y_full)

print(xgb_grid.best_params_)
print(xgb_grid.best_score_)

{'max_depth': 1, 'n_estimators': 15, 'reg_lambda': 0.001}
0.719929203539823


In [7]:
with open('../Data/model.pkl', 'wb') as f:
    dill.dump(xgb_grid, f)